# Context Utilization Benchmark — Google Colab (T4 GPU)

Runs the chunk-deletion ablation pipeline from `chunk_deletion_baseline.py` on a free T4 GPU.

> **Before running:** *Runtime → Change runtime type → T4 GPU*

| Setting | Value |
|---|---|
| Model | `meta-llama/Llama-3.2-1B-Instruct` |
| dtype | `float16` (faster than bfloat16 on T4) |
| Prompt lengths | 512, 1 024, 2 048 tokens |
| Chunk size | 128 tokens |
| Max new tokens | 64 |

Each prompt is a needle-in-a-haystack task — a unique code hidden at the midpoint of a long filler passage.
After the ablation runs, the notebook plots wall time, VRAM usage, and per-chunk influence scores.

In [ ]:
# ── Check GPU ─────────────────────────────────────────────────────────────
import subprocess, torch

out = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv,noheader'],
    capture_output=True, text=True
)
gpu_info = out.stdout.strip()
print('GPU    :', gpu_info or 'NOT FOUND — set Runtime to T4 GPU')
print('PyTorch:', torch.__version__)
print('CUDA   :', torch.cuda.is_available())
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info(0)
    print(f'VRAM   : {total/1e9:.1f} GB total, {free/1e9:.1f} GB free')

if not torch.cuda.is_available():
    raise RuntimeError('No GPU found. Go to Runtime > Change runtime type > T4 GPU.')

In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────
!pip install -q transformers>=4.40 sentence-transformers accelerate numpy matplotlib
print('Dependencies ready.')

In [ ]:
# ── Configuration — edit REPO_URL before running ──────────────────────────
REPO_URL        = 'https://github.com/YOUR_USERNAME/context-utilization'  # <-- set this

MODEL_ID        = 'meta-llama/Llama-3.2-1B-Instruct'
DTYPE           = 'float16'
CHUNK_SIZE      = 128
MAX_NEW_TOKENS  = 64
PROMPT_LENGTHS  = ['512', '1k', '2k']   # ~4, 8, 16 chunks each
NUM_EXAMPLES    = 1
OUTPUT_DIR      = '/content/context-utilization/results'

print('Config set. Remember to update REPO_URL if you have not already.')

In [ ]:
# ── Clone repo (idempotent — safe to re-run) ──────────────────────────────
import os, sys

REPO_PATH = '/content/context-utilization'

if not os.path.exists(REPO_PATH):
    result = subprocess.run(
        ['git', 'clone', REPO_URL, REPO_PATH],
        capture_output=True, text=True
    )
    print(result.stderr or result.stdout)
    if result.returncode != 0:
        raise RuntimeError('git clone failed:\n' + result.stderr)
else:
    print('Repo already present at', REPO_PATH)
    # Pull latest changes
    subprocess.run(['git', '-C', REPO_PATH, 'pull', '--ff-only'],
                   capture_output=True, text=True)

os.chdir(REPO_PATH)
sys.path.insert(0, REPO_PATH)
print('CWD:', os.getcwd())
print('Files:', sorted(f for f in os.listdir('.') if f.endswith('.py')))

In [ ]:
# ── HuggingFace login ─────────────────────────────────────────────────────
# Llama-3.2-1B-Instruct requires accepting Meta's license:
# https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct
#
# Create a token at https://huggingface.co/settings/tokens (read-only is fine).
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# ── Generate needle-in-a-haystack test prompts ────────────────────────────
import json

PROMPTS_FILE = 'test_prompts.jsonl'

cmd = ([sys.executable, 'generate_ruler_prompts.py',
        '--lengths']  + PROMPT_LENGTHS +
       ['--positions', 'middle',
        '--num_examples', str(NUM_EXAMPLES),
        '--output_file', PROMPTS_FILE,
        '--include_metadata'])

result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stderr)
if result.returncode != 0:
    print(result.stdout)
    raise RuntimeError('Prompt generation failed')

with open(PROMPTS_FILE) as f:
    prompts = [json.loads(l) for l in f if l.strip()]

print(f'Generated {len(prompts)} prompts:')
for p in prompts:
    tl  = (p.get('metadata') or {}).get('target_tokens', '?')
    approx = len(p['prompt']) // 4
    ref = p['reference']
    print(f'  target={tl:<6}  approx={approx:<6} tokens  needle={ref}')

In [ ]:
# ── Run chunk-deletion baseline ───────────────────────────────────────────
# Estimated runtime on T4:
#   512 tokens  x 128 chunk  =  5 runs  ~  1-2 min
#   1024 tokens x 128 chunk  =  9 runs  ~  3-4 min
#   2048 tokens x 128 chunk  = 17 runs  ~  6-8 min
#   Total: ~12-15 minutes

os.makedirs(OUTPUT_DIR, exist_ok=True)

cmd = [sys.executable, 'chunk_deletion_baseline.py',
       '--model',          MODEL_ID,
       '--input_file',     PROMPTS_FILE,
       '--chunk_size',     str(CHUNK_SIZE),
       '--max_new_tokens', str(MAX_NEW_TOKENS),
       '--dtype',          DTYPE,
       '--output_dir',     OUTPUT_DIR]

print('Command:', ' '.join(cmd))
print('Running... (progress appears below as each chunk is ablated)')

result = subprocess.run(cmd, check=False)
if result.returncode != 0:
    raise RuntimeError(f'Baseline exited with code {result.returncode}')
print('\nBaseline complete.')

In [ ]:
# ── Load results and print summary table ──────────────────────────────────
import numpy as np

with open(os.path.join(OUTPUT_DIR, 'profiling_summary.json')) as f:
    prof = json.load(f)

with open(os.path.join(OUTPUT_DIR, 'chunk_deletion_results.json')) as f:
    full_results = json.load(f)

print(f'Model      : {prof["model"]}')
print(f'Chunk size : {prof["chunk_size"]} tokens')
print()
hdr = f'{"Tokens":>8}  {"Chunks":>7}  {"Runs":>6}  {"TotalTime(s)":>13}  {"Avg/run(s)":>11}  {"PeakVRAM(MB)":>13}'
print(hdr)
print('-' * len(hdr))
for ex in prof['examples']:
    pt  = ex['prompt_tokens']
    nc  = ex['num_chunks']
    nr  = ex['total_ablation_runs']
    tt  = ex['total_wall_time_s']
    avg = ex['mean_wall_time_s']
    v   = ex['max_peak_vram_mb']
    print(f'{pt:>8}  {nc:>7}  {nr:>6}  {tt:>13.2f}  {avg:>11.3f}  {v:>13.0f}')

In [ ]:
# ── Plot 1: Wall time per ablation run ────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

examples = prof['examples']
n = len(examples)

fig, axes = plt.subplots(1, n, figsize=(6 * n, 4), sharey=False)
if n == 1:
    axes = [axes]

for ax, ex, res in zip(axes, examples, full_results):
    times  = [p['wall_time_s'] for p in res['profile']]
    labels = ['base'] + ['C' + str(i + 1) for i in range(len(times) - 1)]
    colors = ['steelblue'] + ['coral'] * (len(times) - 1)

    ax.bar(range(len(times)), times, color=colors, edgecolor='white', linewidth=0.4)
    mean_t = np.mean(times)
    ax.axhline(mean_t, color='grey', linestyle='--', linewidth=0.9,
               label=f'mean={mean_t:.2f}s')

    pt = ex['prompt_tokens']
    nc = ex['num_chunks']
    ax.set_title(f'{pt} tokens  ({nc} chunks of {prof["chunk_size"]})', fontsize=11)
    ax.set_xlabel('Run')
    ax.set_ylabel('Wall time (s)')
    ax.set_xticks(range(len(times)))
    ax.set_xticklabels(labels, rotation=70, ha='right', fontsize=7)
    ax.legend(fontsize=8)

fig.legend(
    handles=[mpatches.Patch(color='steelblue', label='Baseline'),
             mpatches.Patch(color='coral',     label='Ablation run')],
    loc='upper right', fontsize=9
)
fig.suptitle('Wall Time per Run — Physical Chunk Deletion', fontsize=13)
plt.tight_layout()
plt.savefig('wall_time_per_run.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: wall_time_per_run.png')

In [ ]:
# ── Plot 2: Scaling — total time and VRAM vs prompt length ────────────────
lens      = [ex['prompt_tokens']        for ex in examples]
tot_t     = [ex['total_wall_time_s']    for ex in examples]
avg_t     = [ex['mean_wall_time_s']     for ex in examples]
peak_v    = [ex['max_peak_vram_mb']     for ex in examples]
n_runs    = [ex['total_ablation_runs']  for ex in examples]
xlabels   = [str(l) for l in lens]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

x = np.arange(len(lens))
w = 0.4
ax1.bar(x - w/2, tot_t, width=w, color='steelblue', label='Total wall time')
ax1.bar(x + w/2, avg_t, width=w, color='coral',     label='Avg per run')
for xi, (nr, tt) in enumerate(zip(n_runs, tot_t)):
    ax1.text(xi - w/2, tt + 0.3, str(nr) + ' runs',
             ha='center', fontsize=8, color='steelblue')
ax1.set_xticks(x)
ax1.set_xticklabels(xlabels)
ax1.set_xlabel('Prompt length (tokens)')
ax1.set_ylabel('Time (s)')
ax1.set_title('Wall Time vs Prompt Length')
ax1.legend()

ax2.bar(xlabels, peak_v, color='mediumpurple', edgecolor='white')
for xi, v in enumerate(peak_v):
    ax2.text(xi, v + 30, f'{v:.0f}', ha='center', fontsize=9)
ax2.set_xlabel('Prompt length (tokens)')
ax2.set_ylabel('Peak VRAM (MB)')
ax2.set_title('Peak VRAM vs Prompt Length')

fig.suptitle('Ablation Scaling on T4 — Physical Chunk Deletion', fontsize=13)
plt.tight_layout()
plt.savefig('scaling_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: scaling_summary.png')

In [ ]:
# ── Plot 3: Per-chunk influence scores ────────────────────────────────────
import matplotlib.cm as cm

n = len(full_results)
fig, axes = plt.subplots(1, n, figsize=(6 * n, 5))
if n == 1:
    axes = [axes]

for ax, res, ex in zip(axes, full_results, examples):
    scores = [c['influence_score'] for c in res['chunk_influences']]
    m      = len(scores)
    max_s  = max(scores) if max(scores) > 0 else 1.0
    clabels = ['C' + str(i + 1) for i in range(m)]

    ax.barh(clabels, scores, color=cm.YlOrRd([s / max_s for s in scores]))

    mean_s = np.mean(scores)
    ax.axvline(mean_s, color='steelblue', linestyle='--', linewidth=1.2,
               label=f'mean={mean_s:.4f}')

    pwup = res['pwup']
    eucr = res['eucr']
    pt   = ex['prompt_tokens']
    e10  = eucr.get(0.1, 0)
    b_v  = pwup['B']
    m_v  = pwup['M']
    e_v  = pwup['E']
    ax.set_title(
        f'{pt} tokens   EUCR@10%={e10:.3f}   PWUP B={b_v:.2f} M={m_v:.2f} E={e_v:.2f}',
        fontsize=10
    )
    ax.set_xlabel('Influence score  (1 - cosine similarity)')
    ax.set_xlim(0, max_s * 1.15)
    ax.legend(fontsize=8)

    sm = plt.cm.ScalarMappable(cmap='YlOrRd', norm=plt.Normalize(0, max_s))
    sm.set_array([])
    plt.colorbar(sm, ax=ax, fraction=0.04, pad=0.02)

fig.suptitle('Per-Chunk Influence Scores — Physical Chunk Deletion', fontsize=13)
plt.tight_layout()
plt.savefig('influence_scores.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: influence_scores.png')